# CrowdSky client — 47 Tucanae worked example

Query CrowdSky for stacked frames of the globular cluster **47 Tucanae** (NGC 104), download one
frame and its star table, display the **green channel** of the stack, and overplot circles around
the **Gaia-matched** and **unmatched** SEP detections.

**Prerequisites**

```bash
pip install crowdsky-client matplotlib
export CROWDSKY_API_KEY="csk_..."   # mint at https://crowdsky.univie.ac.at -> Account
```

The client reads `CROWDSKY_API_KEY` from the environment automatically.

In [ ]:
from io import BytesIO
import json
import math

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.visualization import ZScaleInterval, ImageNormalize

from crowdsky_client import Client, CrowdSkyError

# 47 Tucanae / NGC 104
RA_47TUC, DEC_47TUC = 6.0224, -72.0814

client = Client()   # api_key from CROWDSKY_API_KEY; base_url defaults to production

## 1. Find frames covering 47 Tuc

`frames_for_target` cone-searches the HEALPix tiles covering the position (default radius 0.75°,
about the Seestar field of view), unions the per-tile results, and dedups by frame `id`.

In [ ]:
frames = client.frames_for_target(RA_47TUC, DEC_47TUC, radius_deg=0.75)
print(f"{len(frames)} frames cover 47 Tuc")

# Peek at the metadata available per frame.
if frames:
    print("fields:", sorted(frames[0].keys()))
    for f in frames[:5]:
        print(f"  id={f['id']:>6}  {f.get('filter_name','?'):>5}  "
              f"exp={f.get('total_exptime')}s  stars={f.get('n_stars_detected')}  "
              f"{f.get('date_obs_start')}")

## 2. Pick a frame

We take the deepest stack (largest cumulative exposure) as a good target for showing stars.

In [ ]:
if not frames:
    raise SystemExit("No frames returned — check your key and try a wider radius.")

frame = max(frames, key=lambda f: f.get("total_exptime") or 0)
frame_id = frame["id"]
print(f"Using frame {frame_id}: {frame.get('object_name')} "
      f"filter={frame.get('filter_name')} exp={frame.get('total_exptime')}s")

## 3. Download the stacked FITS and extract the green layer

`download_frame` returns the FITS as bytes, which we open straight from memory. Seestar stacks are
one-shot-colour, so the data is typically a 3-plane cube (R, G, B); we take the green plane. (Use
`download_frame_to(frame_id, path)` to stream straight to disk instead.)

In [ ]:
raw = client.download_frame(frame_id)
assert raw[:6] == b"SIMPLE", "expected a FITS file"

with fits.open(BytesIO(raw)) as hdul:
    data = hdul[0].data
print("FITS shape:", None if data is None else data.shape)

def green_layer(arr):
    a = np.asarray(arr, dtype=float)
    if a.ndim == 2:            # mono
        return a
    if a.ndim == 3:
        if a.shape[0] == 3:    # (3, ny, nx)
            return a[1]
        if a.shape[-1] == 3:   # (ny, nx, 3)
            return a[..., 1]
    raise ValueError(f"Unexpected FITS shape {a.shape}")

green = green_layer(data)
print("green layer:", green.shape)

# Optionally keep the file:
# client.download_frame_to(frame_id, f"frame_{frame_id}.fits")

## 4. Download the star table (the "star file")

`star_data` returns the frame's SEP detections cross-matched against Gaia, as JSON. Schemas can
evolve, so we normalise into a list of records and print the available fields.

In [ ]:
stars = client.star_data(frame_id)

# Optionally save the raw star file:
# with open(f"frame_{frame_id}_stars.json", "w") as fh:
#     json.dump(stars, fh)

def to_records(obj):
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        for key in ("stars", "data", "sources"):
            if isinstance(obj.get(key), list):
                return obj[key]
        # column-oriented dict of equal-length lists -> list of row dicts
        cols = {k: v for k, v in obj.items() if isinstance(v, list)}
        if cols:
            n = min(len(v) for v in cols.values())
            return [{k: cols[k][i] for k in cols} for i in range(n)]
    return []

records = to_records(stars)
print(f"{len(records)} detections")
if records:
    print("fields:", sorted(records[0].keys()))

## 5. Split into Gaia-matched vs unmatched

We locate the pixel-coordinate columns and a field indicating a Gaia association. The helpers try a
range of likely names and fall back gracefully — adjust the name lists to your data if needed (the
printed `fields` above tell you what's there).

In [ ]:
sample = records[0] if records else {}

def pick(rec, names):
    for n in names:
        if n in rec:
            return n
    return None

xkey = pick(sample, ["x", "x_pix", "xpix", "xcentroid", "X", "col", "xcenter"])
ykey = pick(sample, ["y", "y_pix", "ypix", "ycentroid", "Y", "row", "ycenter"])

GAIA_FIELDS = ["gaia_source_id", "source_id", "gaia_id", "gaia_g", "gaia_g_mag",
               "phot_g_mean_mag", "gaia_v", "v_mag", "synth_v", "gaia_match",
               "matched", "in_gaia", "has_gaia"]
match_key = pick(sample, GAIA_FIELDS)
print(f"x={xkey!r} y={ykey!r} gaia-match field={match_key!r}")

def is_matched(rec):
    if match_key is None:
        return False
    v = rec.get(match_key)
    if v is None:
        return False
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return not (isinstance(v, float) and math.isnan(v))
    return str(v).strip().lower() not in ("", "0", "false", "none", "nan")

def xy(rec):
    return rec.get(xkey), rec.get(ykey)

matched = [xy(r) for r in records if is_matched(r) and None not in xy(r)]
unmatched = [xy(r) for r in records if not is_matched(r) and None not in xy(r)]
print(f"{len(matched)} Gaia-matched, {len(unmatched)} unmatched")

## 6. Plot the green layer with circled detections

Green circles mark Gaia-matched detections, red circles the unmatched ones (cosmic rays, hot
pixels, blends, or genuine transients). If the circles look offset by ~1 pixel, the star table is
1-indexed (FITS convention) — subtract 1 from the coordinates before plotting.

In [ ]:
norm = ImageNormalize(green, interval=ZScaleInterval())

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(green, origin="lower", cmap="gray", norm=norm)

if matched:
    mx, my = zip(*matched)
    ax.scatter(mx, my, s=90, facecolors="none", edgecolors="lime", linewidths=1.2,
               label=f"Gaia-matched ({len(matched)})")
if unmatched:
    ux, uy = zip(*unmatched)
    ax.scatter(ux, uy, s=90, facecolors="none", edgecolors="red", linewidths=1.2,
               label=f"unmatched ({len(unmatched)})")

ax.set_title(f"47 Tuc — frame {frame_id} (green channel)")
ax.set_xlabel("x [pixels]")
ax.set_ylabel("y [pixels]")
ax.legend(loc="upper right", framealpha=0.8)
plt.tight_layout()
plt.show()

## Next steps

- Filter your query server-side: `frames_by_healpix(pix, filter_name="LP", min_exptime=120)`.
- Stream large frames to disk: `client.download_frame_to(frame_id, "frame.fits")`.
- Survey the whole sky before pulling frames: `coverage = client.sky_coverage()` (an
  `astropy.table.Table`, no key required).

See the [full documentation](https://crowdsky-client.readthedocs.io) for the complete API.